# Camera Calibration

In [ ]:
%pip install -q opencv-contrib-python-headless

from google.colab import drive
drive.mount("/content/drive")

from pathlib import Path
import cv2
import numpy as np

project = Path("/content/drive/MyDrive/Projects/humanoid-challenge")
video_path = project / "data/raw/calibration_1.MOV"

assert video_path.exists(), f"File not found: {video_path}"
print("OpenCV:", cv2.__version__)

In [ ]:
dictionary = cv2.aruco.getPredefinedDictionary(
    cv2.aruco.DICT_4X4_50
)
board = cv2.aruco.CharucoBoard(
    (5, 7), 0.030, 0.022, dictionary
)
detector = cv2.aruco.CharucoDetector(board)

cap = cv2.VideoCapture(str(video_path))
assert cap.isOpened(), "Could not open the video."
cap.set(cv2.CAP_PROP_ORIENTATION_AUTO, 1)

fps = cap.get(cv2.CAP_PROP_FPS)
assert fps > 0
stride = max(1, round(fps))

object_points, image_points = [], []
frame_ids, sharpness_scores = [], []
preview = None
image_size = None
frame_id = 0

while True:
    ok, frame = cap.read()
    if not ok:
        break

    if frame_id % stride == 0:
        gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
        image_size = (gray.shape[1], gray.shape[0])

        corners, ids, _, _ = detector.detectBoard(gray)

        # The board has 24 internal corners; require at least half.
        if ids is not None and len(ids) >= 12:
            obj, img = board.matchImagePoints(corners, ids)
            object_points.append(obj.astype(np.float32))
            image_points.append(img.astype(np.float32))
            frame_ids.append(frame_id)
            sharpness_scores.append(
                cv2.Laplacian(gray, cv2.CV_64F).var()
            )

            if preview is None:
                points = np.asarray(corners).reshape(-1, 2)
                corner_ids = np.asarray(ids).reshape(-1)

                print("Original shapes:", corners.shape, ids.shape)
                assert len(points) == len(corner_ids), (
                    f"Count mismatch: {len(points)} corners, "
                    f"{len(corner_ids)} IDs"
                )

                preview = frame.copy()
                for (x, y), corner_id in zip(points, corner_ids):
                    position = (int(round(x)), int(round(y)))
                    cv2.circle(preview, position, 5, (0, 255, 0), -1)
                    cv2.putText(
                        preview, str(int(corner_id)),
                        (position[0] + 7, position[1] - 7),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.5,
                        (0, 0, 255), 1, cv2.LINE_AA,
                    )

    frame_id += 1

cap.release()

print("Decoded image size (width, height):", image_size)
print("Usable sampled frames:", len(object_points))
assert len(object_points) >= 15, (
    "Too few detected views. Inspect detection before calibrating."
)

import matplotlib.pyplot as plt
plt.figure(figsize=(7, 9))
plt.imshow(cv2.cvtColor(preview, cv2.COLOR_BGR2RGB))
plt.axis("off")
plt.show()

In [ ]:
validation_ids = [
    i for i in range(len(object_points)) if i % 5 == 0
]
training_ids = [
    i for i in range(len(object_points)) if i % 5 != 0
]

rms, camera_matrix, distortion, _, _ = cv2.calibrateCamera(
    [object_points[i] for i in training_ids],
    [image_points[i] for i in training_ids],
    image_size,
    None,
    None,
)

validation_errors = []

for i in validation_ids:
    ok, rvec, tvec = cv2.solvePnP(
        object_points[i], image_points[i],
        camera_matrix, distortion,
    )
    if not ok:
        continue

    projected, _ = cv2.projectPoints(
        object_points[i], rvec, tvec,
        camera_matrix, distortion,
    )
    residual = (
        projected.reshape(-1, 2)
        - image_points[i].reshape(-1, 2)
    )
    validation_errors.append(
        float(np.sqrt(np.mean(np.sum(residual**2, axis=1))))
    )

assert validation_errors, "No validation poses could be estimated."

print("Camera matrix:\n", camera_matrix)
print("Distortion coefficients:", distortion.ravel())
print(f"Training reprojection RMS: {rms:.3f} pixels")
print(
    "Validation RMS, median / maximum:",
    f"{np.median(validation_errors):.3f} / "
    f"{np.max(validation_errors):.3f} pixels",
)

In [ ]:
output_dir = project / "outputs/calibration"
output_dir.mkdir(parents=True, exist_ok=True)

calibration_path = output_dir / "iphone13mini_1080x1920_provisional.npz"

np.savez(
    calibration_path,
    camera_matrix=camera_matrix,
    distortion=distortion,
    image_size=np.array(image_size),
    training_rms=rms,
    validation_errors=np.array(validation_errors),
    board_squares=np.array([5, 7]),
    square_length_m=0.030,
    board_marker_length_m=0.022,
    hand_marker_length_m=0.040,
    dictionary="DICT_4X4_50",
    hand_marker_id=49,
    opencv_version=cv2.__version__,
)

print("Saved:", calibration_path)

In [ ]:
import cv2
import numpy as np
import json
from pathlib import Path

video_path = project / "data/raw/marker_test_01.MOV'"
c = np.load(project/'outputs/calibration/iphone13mini_1080x1920_provisional.npz')

K = c['camera_matrix']
d = c['distortion']
size = c['image_size']

a = cv2.aruco
dictionary = a.getPredefinedDictionary(a.DICT_4X4_50)

board = a.CharucoBoard((5,7),.030,.022,dictionary)
detector = a.CharucoDetector(board)
s = .04
obj = np.array([[-s/2,s/2,0],[s/2,s/2,0],[s/2,-s/2,0],[-s/2,-s/2,0]],np.float32)
cap = cv2.VideoCapture(str(project/'data/raw/marker_test_01.MOV'))
cap.set(cv2.CAP_PROP_ORIENTATION_AUTO,1)

fps = cap.get(cv2.CAP_PROP_FPS)
rows = []
total = 0
detected = 0
i = 0

while True:
    ok, frame = cap.read()

    if not ok:
        break

    if i % 10 == 0:
        total += 1
        assert tuple(size) == (frame.shape[1],frame.shape[0])

    corners, ids, mc, mi = detector.detectBoard(cv2.cvtColor(frame,cv2.COLOR_BGR2GRAY))
    if mi is not None and 49 in mi.flatten():
        detected += 1

        if ids is not None and len(ids) >= 12:
            bp, ip = board.matchImagePoints(corners,ids)
            okb, rb, tb = cv2.solvePnP(bp, ip, K, d)
            m = np.asarray(mc[list(mi.flatten()).index(49)], np.float32).reshape(4,2)
            okm, rm, tm = cv2.solvePnP(obj, m, K, d, flags=cv2.SOLVEPNP_IPPE_SQUARE)

            if okb and okm:
                R = cv2.Rodrigues(rb)[0]
                pos=(R.T@(tm-tb)).flatten()
                proj = cv2.projectPoints(obj, rm, tm, K, d)[0].reshape(4,2)
                err = np.sqrt(np.mean(np.sum((proj-m)**2, axis=1)))
                rows.append([i/fps, *pos, err])
    i+=1

cap.release()
arr = np.array(rows)
np.savetxt('/tmp/marker_tracking.csv', arr,delimiter=',', header='seconds,x_m,y_m,z_m,marker_reprojection_px')
print(json.dumps({'sampled':total, 'marker_detected':detected, 'paired_poses':len(rows), 'median_marker_reprojection_px':float(np.median(arr[:,4])), 'position_min_m':arr[:,1:4].min(axis=0).tolist(), 'position_max_m':arr[:,1:4].max(axis=0).tolist()}))

for start,end in [(1,5),(9,11),(12,14),(17,19),(23,26),(29,32),(40,43)]:
    v=arr[(arr[:,0] >= start) & (arr[:,0] <= end), 1:4]

    if len(v):
        print(start, end, 'median mm', np.round(np.median(v,axis=0)*1000,1), 'std mm', np.round(v.std(axis=0)*1000,1))
